In [2]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

# 1. Initialize clients
target_client = new_client.target
activity_client = new_client.activity

# 2. Get the correct target ID for Human EGFR
print("Searching for target...")
target_query = target_client.search('EGFR')
targets = pd.DataFrame.from_dict(target_query)

# Filter for human EGFR to ensure high-quality data
human_egfr = targets[(targets['organism'] == 'Homo sapiens') & (targets['target_type'] == 'SINGLE PROTEIN')]
target_chembl_id = human_egfr.iloc[0]['target_chembl_id']
print(f"Found Target ID: {target_chembl_id}")

# 3. Fetch bioactivities (Removing .only() fixes the missing data bug)
print("Fetching activities (this may take a minute)...")
res = activity_client.filter(target_chembl_id=target_chembl_id, standard_type="IC50")

# 4. Convert to list of dicts safely
res_list = list(res)

if not res_list:
    print("No records found! Check your filters.")
else:
    # 5. Extract only the columns you need manually to avoid nested dict issues
    extracted_data = []
    for record in res_list:
        extracted_data.append({
            'molecule_chembl_id': record.get('molecule_chembl_id'),
            'canonical_smiles': record.get('molecule_structures', {}).get('canonical_smiles') if record.get('molecule_structures') else None,
            'standard_value': record.get('standard_value'),
            'standard_units': record.get('standard_units'),
            'pchembl_value': record.get('pchembl_value')
        })

    # 6. Save to dataframe and export
    df = pd.DataFrame(extracted_data)
    
    # Drop rows that are missing SMILES strings or IC50 values for clean ML data
    df.dropna(subset=['canonical_smiles', 'standard_value'], inplace=True)
    
    df.to_csv('chembl_egfr_data.csv', index=False)
    print(f"Success! Saved {len(df)} rows to 'chembl_egfr_data.csv'")


Searching for target...
Found Target ID: CHEMBL203
Fetching activities (this may take a minute)...
Success! Saved 0 rows to 'chembl_egfr_data.csv'
